In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
def load_and_prepare_data(file_path, window_size=30):
    """Loads data, scales features, and creates sequences for LSTM."""
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')
    
    # We use 'Close' as the main feature and target
    # You can add more features here like ['Open', 'High', 'Low', 'Volume']
    data = df[['Close']].values
    
    # LSTM is very sensitive to scale, so we normalize to (0, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)
    
    X, y = [], []
    for i in range(window_size, len(scaled_data)):
        # Grab the last 'window_size' days as features (X)
        X.append(scaled_data[i-window_size:i, 0])
        # Grab the current day's price as the label (y)
        y.append(scaled_data[i, 0])
        
    X, y = np.array(X), np.array(y)
    # Reshape X to be [samples, time_steps, features] required for LSTM
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    
    return X, y, scaler

In [ ]:
def split_data(X, y, split_ratio=0.8):
    """Splits data chronologically into train and test sets."""
    split_index = int(len(X) * split_ratio)
    
    X_train, X_test = X[:split_index], X[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]
    
    return X_train, X_test, y_train, y_test

In [ ]:
def build_lstm_model(input_shape):
    """Defines the LSTM architecture."""
    model = Sequential([
        # First LSTM layer with Dropout to prevent overfitting
        LSTM(units=50, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        
        # Second LSTM layer
        LSTM(units=50, return_sequences=False),
        Dropout(0.2),
        
        # Output layer
        Dense(units=1)
    ])
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

In [ ]:
def train_lstm(model, X_train, y_train, epochs=20, batch_size=32):
    """Trains the model over multiple epochs."""
    print(f"Starting training for {epochs} epochs...")
    history = model.fit(
        X_train, y_train, 
        epochs=epochs, 
        batch_size=batch_size, 
        validation_split=0.1, # Use 10% of training data to check errors during training
        verbose=1
    )
    return history

In [ ]:
def evaluate_lstm(model, X_test, y_test, scaler):
    """Predicts and converts values back to original dollar prices."""
    predictions = model.predict(X_test)
    
    # Undo scaling to get actual dollar prices
    predictions = scaler.inverse_transform(predictions)
    y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    mae = mean_absolute_error(y_test_actual, predictions)
    r2 = r2_score(y_test_actual, predictions)
    
    print(f"\nLSTM Model Evaluation:")
    print(f"- Mean Absolute Error: ${mae:.2f}")
    print(f"- R-squared Score: {r2:.4f}")
    
    return y_test_actual, predictions

In [ ]:
def plot_lstm_results(actual, predicted):
    """Visualizes the final predictions vs actual prices."""
    plt.figure(figsize=(12, 6))
    plt.plot(actual[-200:], label='Actual Stock Price', color='blue')
    plt.plot(predicted[-200:], label='Predicted Stock Price', color='red', linestyle='--')
    plt.title('LSTM Prediction: Actual vs Predicted (Last 200 Days)')
    plt.xlabel('Time')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.show()

In [ ]:
if __name__ == "__main__":
    FILE = 'cleaned.csv'
    WINDOW = 60 # look back at the last 60 days to predict tomorrow
    
    X, y, scaler = load_and_prepare_data(FILE, window_size=WINDOW)
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # building &training the LSTM model
    lstm_model = build_lstm_model((X_train.shape[1], 1))
    train_lstm(lstm_model, X_train, y_train, epochs=10, batch_size=64)
    
    # evaluating and visualizing the results
    actual_prices, predicted_prices = evaluate_lstm(lstm_model, X_test, y_test, scaler)
    plot_lstm_results(actual_prices, predicted_prices)